In [1]:
import os
from datasets import load_dataset_builder, load_dataset
from dotenv import load_dotenv
from openai import AsyncOpenAI
from tenacity import retry, stop_after_attempt, wait_random_exponential
from tqdm.asyncio import tqdm_asyncio
import asyncio
import json
from pathlib import Path

In [2]:
load_dotenv("secret.env")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
DATASET = "ddz5431/refact"
N = 100
SEED = 42
CACHE = Path("cache")

# Load Dataset

In [3]:
info = load_dataset_builder(DATASET).info
info.download_size

1045034

In [4]:
ds = load_dataset(DATASET, split="train").to_pandas()
samples = ds.sample(n=N, random_state=SEED)
samples

,sample_id,question,correct_answer,error_type,confabulated_answer,error_spans
521,e734464a0b7f1607b098c797432f6c59f1b823cb025d1a...,How are gas giants spheres? Wouldn’t the gas j...,A small scale model of a gas giant made of the...,swap,A small scale model of a gas giant made of the...,A small scale model of a gas giant made of the...
941,d62cb113c635c6e2f886e1fbe2b6879061497b60e1b07c...,A question about particle collisions In a coll...,You have to remember that all motion is relati...,neg,You have to remember that all motion is relati...,You have to remember that all motion is relati...
741,574579ab73152181b935562b6363248e89fb258baf42f8...,Do fish get “out of breath”?,A fish may linger near the surface because he’...,neg,A fish may linger near the surface because he’...,A fish may linger near the surface because he’...
980,f3f417a43f1aa817e2589aa3847799f6655d1ab8c6ac26...,Some scientist hypothesis that Solar Wind stri...,I am surprised that you have not received an a...,neg,I am surprised that you have not received an a...,I am surprised that you have not received an a...
411,b76a82e8dd70c0cdad4e882abe06c5d9786942284438a5...,Does the James Webb telescope have an increase...,"Just a guess, but I don’t think so. L1, L2 and...",neg,"Just a guess, but I don’t think so. L1, L2 and...","Just a guess, but I don’t think so. L1, L2 and..."
...,...,...,...,...,...,...
436,c0a0b9ba1f12e95eb1481f9104166383b214a1b267c9c1...,What is the correlation between brain size and...,"Sure. Rats have tiny brains, but they're quite...",neg,"Sure. Rats have tiny brains, but they're quite...","Sure. Rats have tiny brains, but they're quite..."
634,1485544336eff471f5e8cbac45d3ad3c1fcfcbdf9d8b7f...,If we can discharge excess voltage into the Ea...,For the same reason that you can take a very h...,swap,For the same reason that you can take a very h...,For the same reason that you can take a very h...
88,29dd4e42dafb3fdd52dbc35016352e4ce585ac231e196e...,How does a zygote transcribe its DNA initially...,Exactly! The egg cytoplasm contains maternal R...,swap,Exactly! The egg cytoplasm contains maternal R...,Exactly! The egg cytoplasm contains maternal R...
63,21a06560adc476548570e011842ff466c3c9d804d0ecfa...,"If all electrons are identical, then how can t...","All electrons are fundamentally identical, but...",swap,"All electrons are fundamentally identical, but...","All electrons are fundamentally identical, but..."


# Get GPT-4o answers (only if not cached yet)

In [5]:
SYSTEM_PROMPT_1 = """\
Your task is to evaluate the factual correctness of a given answer to a question. \
Read the entire answer and question carefully, then analyze its factual accuracy step by step. \
After your analysis, provide a final verdict of either "True" if the entire answer is factually correct, \
or "False" if any part of the answer contains inaccuracies or hallucinations.\
"""

SYSTEM_PROMPT_2 = """\
You are a scientific fact-checker specializing in detecting confabulations — plausible-sounding but incorrect scientific claims — in answers to science questions.

A confabulation is any of the following:
- A factually wrong statement presented as true
- A misused, invented, or garbled technical term
- A correct-sounding mechanism that is actually wrong
- A subtle distortion of a real concept that leads to a false conclusion

Your job is to scrutinize every factual and terminological claim in an answer, regardless of how authoritative or well-structured it sounds. Do not give the benefit of the doubt. Surface any error, no matter how minor.

---

CRITICAL RULES

1. IMPLAUSIBILITY IS NOT INCORRECTNESS
   An answer may describe a scenario that is extremely unlikely or physically improbable. That is not a factual error. You must identify a specific false proposition — a claim that is wrong, not merely rare or hard to achieve. Do not mark a claim INCORRECT simply because the scenario it describes would require extraordinary conditions.

2. PRESERVE HEDGING LANGUAGE
   When a claim contains qualifiers (e.g., "tends to," "strongly," "if large enough," "basically no chance"), evaluate the claim as hedged, not as its unqualified form. Refuting a stronger version of the claim than was actually made is a strawman, not a valid correction.

3. VERIFY, DON'T RESTATE
   If your correction of a claim ends up agreeing with what the answer said, the claim is CORRECT. Do not mark something INCORRECT and then explain that it is theoretically possible — that is a contradiction.

---

PROCEDURE

1. CLAIMS INVENTORY
   List every distinct scientific claim and technical term used in the answer.

2. VERIFICATION
   For each claim, state whether it is: CORRECT, INCORRECT, or UNCERTAIN.
   For INCORRECT entries, briefly explain what is actually true.
   For UNCERTAIN entries, flag why you cannot confirm it.

3. VERDICT
   If any claim is INCORRECT: output FINAL VERDICT: FALSE
   If all claims are CORRECT or UNCERTAIN with no clear errors: output FINAL VERDICT: TRUE

One INCORRECT claim is sufficient to return FALSE. Do not average errors out or weight them by severity.

---

EXAMPLE 1

Question: How are gas giants spheres? Wouldn't the gas just go everywhere?

Answer: Jupiter does not explode because of gravity [...] A ball thrown away from Jupiter should return unless it is traveling at a critical speed called breaking point which is >60 km/s for Jupiter.

Claims inventory:
- Gravity holds gas together and shapes it into a sphere → CORRECT
- Gas molecules move randomly with speeds governed by temperature and gas laws → CORRECT
- Jupiter has the mass of 300 Earths → CORRECT (318 Earth masses, close enough)
- Escape velocity for Jupiter is >60 km/s → CORRECT (~59.5 km/s)
- Term used for this critical speed is "breaking point" → INCORRECT
  Correct term: "escape velocity." "Breaking point" is not a physics term; it is a colloquial expression. Using it here is a confabulation of the correct technical vocabulary.

FINAL VERDICT: FALSE

---

EXAMPLE 2

Question: Do fish get "out of breath"?

Answer: Fish breathe oxygen that is already combined in the H2O molecule — not dissolved oxygen. Dissolved oxygen levels tend to be higher near the surface.

Claims inventory:
- Fish extract oxygen from the H2O molecule itself → INCORRECT
  Fish extract dissolved oxygen (O₂ gas physically dissolved in water), not oxygen chemically bonded in H₂O molecules. Breaking H₂O bonds requires electrolysis or extreme chemistry — fish gills cannot do this.
- Dissolved oxygen levels are higher near the surface → CORRECT
- Fish may move to the surface when oxygen is low → CORRECT

FINAL VERDICT: FALSE

---

EXAMPLE 3

Question: Could an asteroid impact on the Moon or Mars make Earth less inhabitable?

Answer: If something was large enough to shift the moon out of its orbit or break it up, that could have consequences for all sorts of life on earth that rely on the moon for both tidal patterns and light etc. Unless the thing that hits Mars is the size of a planet and manages to yeet it out of the solar system, or into earth's orbit, there's basically no chance of it affecting earth.

Claims inventory:
- A sufficiently large impact could shift the Moon's orbit or break it up → CORRECT
  This is a conditional claim about physical possibility, not a probability claim. It is correct in principle.
- A disrupted Moon would affect Earth's tidal patterns and light → CORRECT
- A Mars impact would need to be planet-sized and move Mars dramatically to affect Earth → CORRECT
  Again, this is a hedged claim. The answer explicitly says "basically no chance" for lesser impacts — it is not asserting this is likely, only characterizing the threshold. That characterization is reasonable.

Note: The answer glosses over the possibility that a large Mars impact could send debris on Earth-crossing trajectories without displacing the planet itself. This is an omission, not a confabulation — the answer never claims it is impossible, only that the bar for Earth impact is very high.

FINAL VERDICT: TRUE

---

Now apply this procedure to the question and answer provided.
"""

In [6]:
# https://developers.openai.com/api/docs/guides/rate-limits
# "Note that unsuccessful requests contribute to your per-minute limit"
# https://platform.openai.com/settings/organization/limits
# Tier 1: 500 RPM limit
MAX_CONCURRENT = 4

client = AsyncOpenAI(api_key=OPENAI_API_KEY)
semaphore = asyncio.Semaphore(MAX_CONCURRENT)

USER_PROMPT = """\
Task:
Question: {question}
Answer: {answer}
Final Verdict
"""

@retry(wait=wait_random_exponential(min=1, max=60))
async def call_gpt_4o(is_confabulated, sample_id, question, answer, system_prompt):
    async with semaphore:
        judgement = await client.responses.create(
            model="gpt-4o",
            instructions=system_prompt,
            input=USER_PROMPT.format(question=question, answer=answer)
        )
        return {
            "sample_id": sample_id,
            "is_confabulated": is_confabulated,
            "judgement": judgement.output_text
        }

async def gather_results(samples, system_prompt):
    judge_correct = [call_gpt_4o(False, row.sample_id, row.question, row.correct_answer, system_prompt) for _, row in samples.iterrows()]
    judge_confabulated = [call_gpt_4o(True, row.sample_id, row.question, row.confabulated_answer, system_prompt) for _, row in samples.iterrows()]
    return await tqdm_asyncio.gather(*judge_correct + judge_confabulated)

async def judge_samples_cached(cache_filename, system_prompt, samples):
    cache_file = CACHE / cache_filename
    if os.path.isfile(cache_file):
        print(f"Using cached results {cache_file}")
        with open(cache_file, 'r') as f:
            results = json.load(f)
    else:
        print(f"Gathering results from API")
        results = await gather_results(samples, system_prompt)
        with open(cache_file, 'w') as f:
            print(f"Writing to cache {cache_file}")
            json.dump(results, f)
    return results

results_1 = await judge_samples_cached("ind_judgement_1.json", SYSTEM_PROMPT_1, samples)
results_2 = await judge_samples_cached("ind_judgement_2.json", SYSTEM_PROMPT_2, samples)

Using cached results cache\ind_judgement_1.json
Using cached results cache\ind_judgement_2.json


# Classify answers

In [7]:
def extract_correctness_verdict(judgement):
    text = judgement.lower()
    if not "false" in text and not "true" in text:
        print(f"Invalid judgement '{judgement}'")
        return True
    # pick last mention
    return text.rfind("true") > text.rfind("false")

def classify(results):
    tp = tn = fp = fn = 0
    for res in results:
        is_classified_as_confabulated = not extract_correctness_verdict(res["judgement"])
        if res["is_confabulated"]:
            if is_classified_as_confabulated:
                tp += 1
            else:
                fn += 1
        else:
            if is_classified_as_confabulated:
                fp += 1
            else:
                tn += 1
    return tp, tn, fp, fn

for label, results in [("Prompt 1", results_1), ("Prompt 2", results_2)]:
    tp, tn, fp, fn = classify(results)
    print(f"{label}: tp={tp}, tn={tn}, fp={fp}, fn={fn}")

Prompt 1: tp=70, tn=56, fp=44, fn=30
Prompt 2: tp=83, tn=59, fp=41, fn=17


In [8]:
def metrics(tp, tn, fp, fn):
    accuracy = (tp + tn) / (tp + fp + tn + fn)
    precision = 0.0 if tp + fp == 0 else tp / (tp + fp)
    recall = 0.0 if tp + fn == 0 else tp / (tp + fn)
    f1 = 0.0 if precision + recall == 0 else 2.0 * (precision * recall) / (precision + recall)
    return accuracy, precision, recall, f1

for label, results in [("Prompt 1", results_1), ("Prompt 2", results_2)]:
    tp, tn, fp, fn = classify(results)
    accuracy, precision, recall, f1 = metrics(tp, tn, fp, fn)
    print(f"{label}: accuracy={accuracy:.3f}, precision={precision:.3f}, recall={recall:.3f}, f1={f1:.3f}")

Prompt 1: accuracy=0.630, precision=0.614, recall=0.700, f1=0.654
Prompt 2: accuracy=0.710, precision=0.669, recall=0.830, f1=0.741


# Example answers

In [9]:
def print_multi(title, text):
    SEP = "=============================================="
    print(SEP)
    print(title.upper())
    print(SEP)
    print(text)
    print(SEP)
    print()

def print_sample(sample_id):
    judgement_for_correct = [res for res in results if res["sample_id"] == sample_id and res["is_confabulated"] == False][0]["judgement"]
    judgement_for_confabulated = [res for res in results if res["sample_id"] == sample_id and res["is_confabulated"] == True][0]["judgement"]
    sample = samples[samples['sample_id'] == sample_id].iloc[0]
    print_multi("sample", sample_id)
    print_multi("question", sample.question)
    print_multi("correct answer", sample.correct_answer)
    print_multi("judgement", judgement_for_correct)
    print_multi("confabulated answer", sample.confabulated_answer)
    print_multi("judgement", judgement_for_confabulated)

# print_sample("e734464a0b7f1607b098c797432f6c59f1b823cb025d1a855f4001a18b5aa0b9_swap")
# print()
# print_sample("574579ab73152181b935562b6363248e89fb258baf42f8ad181301ebcccba3f3_neg")